<div class="alert alert-info">
    <strong>导航</strong>
    <ul>
        <li>上一节： <a href="9_21_short_spacing_mosaic_joint_imaging.ipynb">9.21 短间距、Mosaicking 与联合成像</a></li>
        <li>下一节： <a href="9_23_observing_design_and_archive_reanalysis.ipynb">9.23 观测设计与归档数据再分析</a></li>
    </ul>
</div>


## 9.22 宽带宽场算法边界：MT-MFS、w-term、A-projection 与方向相关校准

第 9.8 节已经讨论宽带成像中的 MFS、频谱指数和主波束校正，第 9.12 节已经引入 w-term、faceting 和 PB-aware mosaic。本节进一步解释这些算法为什么需要出现，以及它们各自的边界在哪里。现代宽带宽场成像并不是把窄带小视场 CLEAN 简单运行在更大的图像上，而是在频率、方向、时间和极化维度上不断修正“天空是否相同、仪器是否相同、相位中心附近近似是否仍然成立”这些假设。

小视场窄带成像常隐含三个近似：天空亮度在频带内不变；主波束在视场内和频率上可以忽略或事后简单修正；非共面基线项足够小，可以把测量方程看成二维 Fourier 变换。宽带、宽场、低频和高动态范围观测会同时破坏这些近似。算法的目标不是消灭复杂性，而是把复杂性放入可控制的模型中，并让模型复杂度与数据约束相匹配。


### 9.22.1 宽带天空不是同一张图

真实天空亮度依赖频率。对连续谱源，常见近似是

$$
I(\nu)=I_0\left(\frac{\nu}{\nu_0}\right)^\alpha,
$$

其中 $\alpha$ 是谱指数。若存在同步辐射老化、自由-自由吸收、多个源成分混合或热尘埃贡献，单一 power law 可能不够，需要 curvature 或更复杂的谱模型。MT-MFS 的思想是在参考频率附近用 Taylor 展开表示频率行为，例如

$$
I(l,m,\nu)\simeq \sum_{t=0}^{N_t-1} I_t(l,m)\left(\frac{\nu-\nu_0}{\nu_0}\right)^t.
$$

这个表达把宽带合成孔径和谱信息同时纳入成像。$I_0$ 近似对应参考频率强度，后续 Taylor 系数可用于估计谱指数和曲率。但 Taylor 展开是局部近似，依赖频带宽度、信噪比、uv 覆盖随频率变化的方式，以及同一像素内是否真的由单一谱成分主导。


![MT-MFS Taylor limits](figures/practical_wideband_mtmfs_taylor_limits.png)

图中比较了 power law、带曲率谱和 broken spectrum。两项 Taylor 模型可以较好描述平滑 power law 附近的行为，但在宽频带、谱弯曲或谱断裂情形下会留下结构化残差。增加 Taylor 项并不自动解决问题，因为高阶项需要更高信噪比和更好的频率覆盖；若主波束、uv 覆盖或 calibration 误差没有同时处理，高阶项还可能吸收仪器系统误差。


### 9.22.2 频变主波束会制造假谱指数

干涉阵看到的不是纯天空 $I(l,m,\nu)$，而是主波束加权后的表观天空

$$
I_{\rm app}(l,m,\nu)=A(l,m,\nu)I(l,m,\nu).
$$

主波束通常随频率升高而变窄。因此，离相位中心较远的源在高频处会被更强衰减。若直接用未充分处理的表观强度估计谱指数，源会显得比真实情况更陡。主波束校正可以恢复一阶通量尺度，但也会放大边缘噪声，并把主波束模型误差转化为谱指数误差。

宽带 primary beam correction 的难点在于它不是一个单一标量。主波束随频率、偏振、时间、天线、指向误差和方向变化。对于高动态范围宽带图像，先做 MT-MFS 再用一个平均主波束修正，和在成像过程中使用 A-projection 或 wideband primary beam 模型，可能给出不同的频谱指数图。频谱指数图的可信区域通常应比总强度图更保守。


![PB spectral index bias](figures/practical_wideband_pb_spectral_index_bias.png)

左图显示频变主波束如何把本来固定的谱指数变成随半径变陡的表观谱指数。右图显示主波束校正的代价：越靠近高频主波束边缘，噪声放大越严重。宽带图像中，谱指数异常不应马上解释为天体物理结构，必须先排除 primary beam、uv coverage 和 calibration 的频率相关误差。


### 9.22.3 w-term：宽场成像不是严格二维 Fourier 变换

非共面基线的测量方程可写为

$$
V(u,v,w)=\int\frac{I(l,m)}{n}\exp\{-2\pi i[ul+vm+w(n-1)]\}\,dl\,dm,
$$

其中 $n=\sqrt{1-l^2-m^2}$。小视场时 $n\simeq 1$，$w(n-1)$ 项可以忽略；宽场时近似不再成立。对小角度 $\theta$，相位误差量级约为

$$
\Delta\phi_w\sim \pi w\theta^2.
$$

这说明 w-term 随视场半径的平方增长，也随非共面基线坐标 $w$ 增长。低频阵列、宽视场单 pointing、长时间合成和远离相位中心的亮源都会放大这个问题。


![w-term algorithms](figures/practical_widefield_wterm_algorithm_comparison.png)

图中左侧展示 w-term 相位误差随视场半径快速增长。右侧概括三类常用处理方式：w-projection 在 visibility gridding 时使用 w-dependent 卷积核；w-stacking 把数据按 w 分层成像再合并；faceting 把大视场分成多个局部小视场，在各自切平面上近似二维成像。实际成像器常把这些思想组合使用。选择哪种方法，取决于视场、w 分布、动态范围目标、计算资源和方向相关误差是否也需要同时处理。


### 9.22.4 A-projection 与 AW-projection：把方向相关仪器响应放进成像算子

w-term 处理的是几何非共面效应；A-projection 处理的是方向相关的天线响应，例如 primary beam、孔径照明、偏振泄漏和时间变化的 beam。若测量方程中包含方向相关 Jones 项 $A(l,m,\nu,t)$，那么成像时把它留到最后做简单除法，往往不足以达到高动态范围或高精度谱指数要求。A-projection 的思想是在 gridding/degridding 中使用包含孔径响应的卷积核，使模型 visibilities 在预测阶段就带有方向相关仪器响应。

AW-projection 同时处理 A-term 和 w-term。它的优势是统一；代价是卷积核更大、计算更重，并且强依赖 beam 模型是否准确。若 beam 模型不可靠，复杂算法可能把错误模型以更高置信度写入图像。对于低频阵列，A-term 还可能与电离层方向相关相位屏同时出现；对于高频阵列，pointing error 和大气相位变化可能成为主导。算法名称相同，并不代表误差物理相同。


![AW projection DDE](figures/practical_widefield_aw_projection_dde.png)

图中从真实天空、primary beam、表观天空到 PB-corrected 图像展示方向相关效应的基本逻辑。主波束既改变通量尺度，也改变噪声；方向相关相位或 beam 误差会把源结构变成位置相关残差。A/AW-projection 的目标是在成像算子中描述这些效应，而不是在最终图像上只做一次后处理修补。


### 9.22.5 方向相关自校准：自由度与过拟合

方向相关自校准包括 peeling、facet calibration、screen-based calibration、killMS/DDFacet 类方法和其他 3GC 技术。它们的共同思想是：不同天空方向上的残差不能再用一个全场统一增益解释，需要为不同方向求解不同校准项。低频电离层相位屏、宽场 primary beam 误差、离轴亮源、指向误差和大气相位起伏，都可能要求这种处理。

但方向相关校准的自由度增长很快。若把天空分成 $N$ 个方向，每个方向、每个天线、每个时间和频率解一个复增益，参数量会迅速接近甚至超过数据真正能约束的量。过多自由度会吸收真实源结构，压低噪声估计，制造假扩展结构，或把 calibration artefact 写入科学图像。因此，方向相关校准必须伴随保守的解间隔、足够的 S/N、独立验证和残差检查。能降低残差并不等于得到更真实的天空。


![DD calibration overfit](figures/practical_direction_dependent_calibration_overfit.png)

左图用示意曲线说明方向数增加时的两种趋势：未建模方向相关误差下降，但噪声拟合和过拟合风险上升。右图比较低频和高频常见 DDE 的相对重要性。低频常由电离层和宽场 beam 主导；高频常由 pointing、primary beam、tropospheric phase 和快速相位变化主导。处理策略应由误差物理决定，而不是由软件默认参数决定。


### 9.22.6 与真实软件工作流的对应

在 CASA 中，MT-MFS、wideband primary beam correction、w-projection、mosaic gridding 和 A-projection 对应 `tclean` 中不同参数和 gridder 选择。WSClean、DDFacet、IDG、DP3、DPPP、killMS 等工具也覆盖宽带宽场和方向相关处理的不同层级。不同工具之间最重要的差异通常不是命令名称，而是测量方程中包含了哪些项、哪些项被近似、哪些项通过外部 beam 或 sky model 提供。

一个宽带宽场成像结果至少应报告：频带范围和参考频率；MT-MFS Taylor 项数及谱指数/曲率图的掩膜；是否使用 wideband PB correction；成像 gridder、w-term 处理方法和关键参数；primary beam 或 aperture 模型来源；是否做 A/AW-projection；是否做方向相关校准及解区、解间隔和 sky model；图像中哪些区域的谱指数、偏振或低面亮度结构被认为可信；残差是否按方向、频率和时间检查。高动态范围图像尤其需要保留失败案例和参数日志，因为算法复杂度本身就是结果的一部分。


### 9.22.7 本节结论

宽带宽场成像的核心挑战是多个近似同时失效：天空随频率变化，主波束随频率和方向变化，基线不再共面，校准误差也随方向变化。MT-MFS、w-projection、faceting、A/AW-projection 和方向相关自校准分别处理这些失效的不同部分。它们不是互相替代的魔法按钮，而是测量方程中不同项的近似实现。

因此，宽带宽场图像的可信度来自两件事：模型包含了主要物理和仪器效应，同时模型自由度没有超过数据约束。教材训练的目标不是记住某个成像器参数组合，而是能判断一个图像中的谱指数结构、远离相位中心的伪影、方向相关残差和高动态范围限制，分别对应测量方程中的哪一项。
